# Advanced Analytics
This notebook computes VaR/CVaR, rolling Sharpe, investor cohorts, SIP continuity, sector HHI, and includes insights. Deliverables generated: `var_cvar_report.csv`, `rolling_sharpe_chart.png`, `recommender.py`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
DATA_ROOT = Path('data/processed')
RAW_ROOT = Path('data/raw')

In [ ]:
# Read inputs
nav = pd.read_csv(DATA_ROOT / 'cleaned_02_nav_history.csv', parse_dates=['date'])
tx = pd.read_csv(DATA_ROOT / 'cleaned_08_investor_transactions.csv', parse_dates=['transaction_date'])
holdings = pd.read_csv(DATA_ROOT / 'cleaned_09_portfolio_holdings.csv', parse_dates=['portfolio_date'])
scheme_perf = pd.read_csv(RAW_ROOT / '07_scheme_performance.csv')

In [ ]:
# Compute daily returns per fund
nav = nav.sort_values(['amfi_code','date']).copy()
nav['return'] = nav.groupby('amfi_code')['nav'].pct_change()
# VaR (95%) -> 5th percentile, CVaR -> mean of returns <= VaR
def var_cvar(series, pct=5):
    s = series.dropna()
    if len(s)==0:
        return pd.Series({'VaR_95': np.nan, 'CVaR_95': np.nan})
    var = np.nanpercentile(s, pct)
    cvar = s[s <= var].mean() if (s <= var).any() else s.mean()
    return pd.Series({'VaR_95': var, 'CVaR_95': cvar})
var_cvar = nav.groupby('amfi_code')['return'].apply(var_cvar).reset_index()
var_cvar.to_csv('var_cvar_report.csv', index=False)
print('Saved var_cvar_report.csv with', len(var_cvar), 'rows')

In [ ]:
# Rolling 90-day Sharpe for 5 key funds (choose top 5 by aggregated holdings market value)
top_funds = holdings.groupby('amfi_code')['market_value_cr'].sum().nlargest(5).index.tolist()
ret_pivot = nav.pivot(index='date', columns='amfi_code', values='return')
window = 90
rolling_mean = ret_pivot[top_funds].rolling(window).mean()
rolling_std = ret_pivot[top_funds].rolling(window).std()
rolling_sharpe = rolling_mean / rolling_std * np.sqrt(252)
plt.figure(figsize=(12,6))
for code in top_funds:
    plt.plot(rolling_sharpe.index, rolling_sharpe[code], label=str(code))
plt.legend(title='amfi_code')
plt.title('Rolling 90-day Sharpe (annualized)')
plt.xlabel('Date')
plt.ylabel('Sharpe')
plt.grid(True)
plt.tight_layout()
plt.savefig('rolling_sharpe_chart.png')
print('Saved rolling_sharpe_chart.png')

In [ ]:
# Investor cohort analysis
tx2 = tx.copy()
tx2['first_tx'] = tx2.groupby('investor_id')['transaction_date'].transform('min')
tx2['cohort_year'] = tx2['first_tx'].dt.year
# Average SIP amount per cohort (consider SIP transactions only)
sip = tx2[tx2['transaction_type'].str.upper()=='SIP']
cohort_stats = sip.groupby('cohort_year').agg(
    avg_sip_amount = ('amount_inr','mean'),
    total_invested = ('amount_inr','sum')
).reset_index()
# top fund preference per cohort (by sum amount)
top_pref = sip.groupby(['cohort_year','amfi_code'])['amount_inr'].sum().reset_index()
top_pref = top_pref.sort_values(['cohort_year','amount_inr'], ascending=[True,False]).groupby('cohort_year').first().reset_index()
top_pref = top_pref.rename(columns={'amfi_code':'top_amfi_code','amount_inr':'top_amount'})
cohort_summary = cohort_stats.merge(top_pref, on='cohort_year', how='left')
cohort_summary.to_csv('investor_cohort_summary.csv', index=False)
print('Saved investor_cohort_summary.csv')

In [ ]:
# SIP continuity analysis
sip_by_investor = sip.sort_values(['investor_id','transaction_date']).groupby('investor_id')
def avg_gap_days(dates):
    if len(dates)<2:
        return np.nan
    gaps = np.diff(dates.astype('datetime64[D]')).astype(int)
    return gaps.mean()
continuity = sip_by_investor['transaction_date'].apply(lambda x: avg_gap_days(x.values)).reset_index().rename(columns={'transaction_date':'avg_gap_days'})
continuity['n_sip'] = sip_by_investor.size().values
continuity = continuity[continuity['n_sip']>=6].copy()
continuity['at_risk'] = continuity['avg_gap_days'] > 35
continuity.to_csv('sip_continuity.csv', index=False)
print('Saved sip_continuity.csv with', len(continuity), 'investors (6+ SIPs)')

In [ ]:
# Sector HHI concentration
hold = holdings.copy()
# compute HHI per fund (weights in percent)
hhi = hold.groupby('amfi_code').apply(lambda g: ( (g['weight_pct']/100.0)**2 ).sum()).reset_index().rename(columns={0:'HHI'})
# join category from scheme_perf if available
hhi = hhi.merge(scheme_perf[['amfi_code','category','risk_grade']], on='amfi_code', how='left')
hhi.sort_values('HHI', ascending=False).head(10).to_csv('top_hhi_funds.csv', index=False)
print('Saved top_hhi_funds.csv')

**Insights (5)**
1. Funds with highest VaR (most negative 5th percentile) - see `var_cvar_report.csv`.
2. Investor cohort with highest total invested - see `investor_cohort_summary.csv`.
3. SIP continuity: investors with avg gap > 35 days flagged as at-risk in `sip_continuity.csv`.
4. Top concentrated portfolios by HHI are in `top_hhi_funds.csv`.
5. Rolling Sharpe trends for top funds saved as `rolling_sharpe_chart.png`.